# AgeLens — 10 Survey-Weighted Mortality Survival Analysis

This notebook executes and validates the mortality models authorized by D-015.

## Primary model

- Outcome: all-cause mortality
- Time: `PERMTH_EXM`
- Exposure: cycle-specific weighted canonical Phenotypic Age acceleration
- Reporting scale: hazard ratio per 5-year higher acceleration
- Adjustment: chronological age, sex, race/ethnicity, and NHANES cycle
- Design: pooled fasting weight, cycle-unique strata, and cycle-unique PSUs
- Estimator: R `survey::svycoxph`

## Required models

1. Canonical, exposure-only
2. Canonical, primary adjusted
3. Canonical, adjusted, per weighted SD
4. No-topcode sensitivity
5. Erratum sensitivity
6. Creatinine `+0.11 mg/dL` sensitivity
7. Creatinine `+0.17 mg/dL` sensitivity
8. Creatinine `+0.23 mg/dL` sensitivity
9. Exclusion of deaths within 12 months

## Release behavior

Mortality results remain blocked unless every model and validation check passes. Cause-specific mortality remains unauthorized.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
NOTEBOOK_BUILD = "10-v8-r-string-concat-fixed"
DECISION_ID = "D-016"
RELEASE_DATE = "2026-07-22"
RELEASE_MARKER = (
    "<!-- AGE-LENS MORTALITY RESULTS RELEASE 2026-07-22 -->"
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

print(f"Notebook build: {NOTEBOOK_BUILD}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


Notebook build: 10-v8-r-string-concat-fixed
numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


In [2]:
def find_project_root(
    folder_name: str = PROJECT_FOLDER_NAME,
) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'."
    )


def find_rscript() -> Path:
    direct = shutil.which("Rscript")

    if direct:
        return Path(direct).resolve()

    candidates = []

    if os.name == "nt":
        for base in [
            Path(r"C:\Program Files\R"),
            Path(r"C:\Program Files (x86)\R"),
        ]:
            if base.exists():
                candidates.extend(
                    base.glob(r"R-*\bin\Rscript.exe")
                )
                candidates.extend(
                    base.glob(r"R-*\bin\x64\Rscript.exe")
                )

    candidates = sorted(
        {
            path.resolve()
            for path in candidates
            if path.exists()
        },
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            "Rscript was not found. Add R to PATH or install R."
        )

    return candidates[0]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def replace_document_field(
    text: str,
    field: str,
    value: str,
) -> str:
    pattern = re.compile(
        rf"(?m)^(\|\s*{re.escape(field)}\s*\|\s*)"
        rf"([^|]*?)(\s*\|)$"
    )

    updated, count = pattern.subn(
        lambda match: (
            f"{match.group(1)}{value}{match.group(3)}"
        ),
        text,
        count=1,
    )

    if count != 1:
        raise RuntimeError(
            f"Could not update document field: {field}"
        )

    return updated


def extract_document_version(text: str) -> str:
    match = re.search(
        r"(?m)^\|\s*Version\s*\|\s*([^|]+?)\s*\|$",
        text,
    )

    if not match:
        raise RuntimeError(
            "Could not extract document version."
        )

    return match.group(1).strip()


def add_revision_row(
    text: str,
    row: str,
) -> str:
    marker = (
        "*This file is a single authoritative document"
    )
    marker_index = text.find(marker)

    if marker_index < 0:
        raise RuntimeError(
            "Authoritative-document marker was not found."
        )

    before = text[:marker_index].rstrip()
    after = text[marker_index:]

    if row in before:
        return text

    return before + "\n" + row + "\n\n" + after


def dataframe_to_markdown(
    frame: pd.DataFrame,
) -> str:
    columns = [str(column) for column in frame.columns]
    header = "| " + " | ".join(columns) + " |"
    separator = (
        "| "
        + " | ".join(["---"] * len(columns))
        + " |"
    )
    rows = []

    for _, row in frame.iterrows():
        values = []

        for value in row:
            if pd.isna(value):
                text = ""
            elif isinstance(
                value,
                (float, np.floating),
            ):
                text = f"{float(value):.6g}"
            else:
                text = str(value)

            values.append(
                text.replace("|", "\\|")
            )

        rows.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(
        [header, separator, *rows]
    )


def parse_r_boolean(
    series: pd.Series,
) -> pd.Series:
    mapping = {
        "TRUE": True,
        "FALSE": False,
        "T": True,
        "F": False,
        "1": True,
        "0": False,
    }

    parsed = (
        series.astype(str)
        .str.strip()
        .str.upper()
        .map(mapping)
    )

    if parsed.isna().any():
        bad_values = sorted(
            series.loc[
                parsed.isna()
            ].astype(str).unique().tolist()
        )
        raise ValueError(
            f"Could not parse R Boolean values: {bad_values}"
        )

    return parsed.astype(bool)


PROJECT_ROOT = find_project_root()
RSCRIPT_PATH = find_rscript()

CONFIG_PATH = (
    PROJECT_ROOT / "configs" / "agelens_config.json"
)
DECISION_LOG_PATH = (
    PROJECT_ROOT / "docs" / "governance" / "Decision_Log.md"
)
PROTOCOL_PATH = (
    PROJECT_ROOT
    / "docs"
    / "methodology"
    / "Mortality_Analysis_Protocol.md"
)
COHORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "agelens_v1_mortality_cohort_authorized.parquet"
)

required_paths = [
    CONFIG_PATH,
    DECISION_LOG_PATH,
    PROTOCOL_PATH,
    COHORT_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        f"Required mortality-model inputs are missing: "
        f"{missing_paths}"
    )

CONFIG = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

INTERIM_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["interim_data"]
)
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
SCRIPTS_ROOT = PROJECT_ROOT / "scripts"
REPORT_ROOT = (
    PROJECT_ROOT / "docs" / "methodology"
)

for path in [
    INTERIM_ROOT,
    TABLES_ROOT,
    LOGS_ROOT,
    SCRIPTS_ROOT,
    REPORT_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Rscript: {RSCRIPT_PATH}")
print(f"Authorized cohort: {COHORT_PATH}")


Project root: <PROJECT_ROOT>
Rscript: C:\Program Files\R\R-4.5.1\bin\x64\Rscript.exe
Authorized cohort: <PROJECT_ROOT>\data\processed\agelens_v1_mortality_cohort_authorized.parquet


## 1. Verify D-015 and mortality release prerequisites

In [3]:
release_gates = CONFIG.get(
    "release_gates",
    {},
)

required_true_gates = [
    "mortality_analysis_authorized",
    "mortality_cohort_defined",
    "canonical_outputs_regenerated",
    "canonical_regression_checks_passed",
]

for gate in required_true_gates:
    if not release_gates.get(gate, False):
        raise RuntimeError(
            f"Required mortality-model gate is not open: {gate}"
        )

required_false_gates = [
    "mortality_models_completed",
    "mortality_model_validation_passed",
    "mortality_results_allowed",
]

for gate in required_false_gates:
    if release_gates.get(gate, False):
        raise RuntimeError(
            f"Mortality release gate is already open: {gate}"
        )

mortality_config = CONFIG.get(
    "mortality_analysis",
    {},
)

if mortality_config.get("decision") != "D-015":
    raise RuntimeError(
        "D-015 is not recorded as the mortality authorization."
    )

if mortality_config.get(
    "cause_specific_mortality_authorized",
    True,
):
    raise RuntimeError(
        "Cause-specific mortality is unexpectedly authorized."
    )

decision_text = DECISION_LOG_PATH.read_text(
    encoding="utf-8"
)
protocol_text = PROTOCOL_PATH.read_text(
    encoding="utf-8"
)

if extract_document_version(decision_text) != "1.8":
    raise RuntimeError(
        "Decision Log must be version 1.8 before D-016."
    )

if extract_document_version(protocol_text) != "1.0":
    raise RuntimeError(
        "Mortality protocol must be version 1.0 before "
        "model implementation."
    )

if "### D-015" not in decision_text:
    raise RuntimeError(
        "D-015 is missing from the Decision Log."
    )

if f"### {DECISION_ID}" in decision_text:
    raise RuntimeError(
        f"{DECISION_ID} already exists in the Decision Log."
    )

if RELEASE_MARKER in protocol_text:
    raise RuntimeError(
        "The mortality results release marker already exists."
    )

print("✅ D-015 and mortality release prerequisites verified.")


✅ D-015 and mortality release prerequisites verified.


## 2. Validate and export the authorized model input

In [4]:
cohort = pd.read_parquet(
    COHORT_PATH
)

required_columns = {
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "RIAGENDR",
    "RIDRETH3",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "pooled_stratum",
    "pooled_psu",
    "phenoage_acceleration_years",
    "phenoage_acceleration_per_5_years",
    "phenoage_acceleration_per_sd",
    "sensitivity_erratum_acceleration_years",
    "sensitivity_creatinine_plus_0_11_acceleration_years",
    "sensitivity_creatinine_plus_0_17_acceleration_years",
    "sensitivity_creatinine_plus_0_23_acceleration_years",
    "mortality_event",
    "followup_months",
    "early_death_within_12_months",
    "no_topcode_sensitivity_eligible",
    "early_death_exclusion_eligible",
    "mortality_analysis_authorized",
    "mortality_results_allowed",
    "governing_decision",
}

missing_columns = sorted(
    required_columns - set(cohort.columns)
)

if missing_columns:
    raise ValueError(
        "Authorized cohort is missing required columns: "
        f"{missing_columns}"
    )

if cohort.empty:
    raise RuntimeError(
        "Authorized mortality cohort is empty."
    )

if cohort.duplicated(
    ["NHANES_CYCLE", "SEQN"]
).any():
    raise RuntimeError(
        "Duplicate cycle + SEQN rows exist in the cohort."
    )

if not cohort[
    "mortality_analysis_authorized"
].fillna(False).astype(bool).all():
    raise RuntimeError(
        "One or more cohort rows are not authorized for "
        "mortality analysis."
    )

if cohort[
    "mortality_results_allowed"
].fillna(False).astype(bool).any():
    raise RuntimeError(
        "The participant file already marks mortality results "
        "as reportable."
    )

if not cohort[
    "governing_decision"
].eq("D-015").all():
    raise RuntimeError(
        "The cohort does not consistently reference D-015."
    )

finite_columns = [
    "chronological_age_years",
    "WTSAF4YR",
    "phenoage_acceleration_years",
    "phenoage_acceleration_per_5_years",
    "phenoage_acceleration_per_sd",
    "sensitivity_erratum_acceleration_years",
    "sensitivity_creatinine_plus_0_11_acceleration_years",
    "sensitivity_creatinine_plus_0_17_acceleration_years",
    "sensitivity_creatinine_plus_0_23_acceleration_years",
    "followup_months",
]

if not np.isfinite(
    cohort[finite_columns].to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "The authorized model input contains non-finite values."
    )

if not cohort["mortality_event"].isin(
    [0, 1]
).all():
    raise RuntimeError(
        "Mortality event is not binary."
    )

if cohort["WTSAF4YR"].le(0).any():
    raise RuntimeError(
        "The mortality cohort contains non-positive weights."
    )

if cohort["followup_months"].le(0).any():
    raise RuntimeError(
        "The mortality cohort contains non-positive follow-up."
    )

cohort["sensitivity_erratum_acceleration_per_5_years"] = (
    cohort[
        "sensitivity_erratum_acceleration_years"
    ]
    / 5.0
)
cohort[
    "sensitivity_creatinine_plus_0_11_acceleration_per_5_years"
] = (
    cohort[
        "sensitivity_creatinine_plus_0_11_acceleration_years"
    ]
    / 5.0
)
cohort[
    "sensitivity_creatinine_plus_0_17_acceleration_per_5_years"
] = (
    cohort[
        "sensitivity_creatinine_plus_0_17_acceleration_years"
    ]
    / 5.0
)
cohort[
    "sensitivity_creatinine_plus_0_23_acceleration_per_5_years"
] = (
    cohort[
        "sensitivity_creatinine_plus_0_23_acceleration_years"
    ]
    / 5.0
)

MODEL_INPUT_PATH = (
    INTERIM_ROOT
    / "10_mortality_model_input.csv"
)

cohort.to_csv(
    MODEL_INPUT_PATH,
    index=False,
    na_rep="",
)

def sample_summary(
    sample: str,
    mask: pd.Series,
) -> dict[str, object]:
    subset = cohort.loc[
        mask.fillna(False).astype(bool)
    ]

    return {
        "sample": sample,
        "n": int(len(subset)),
        "events": int(
            subset["mortality_event"].sum()
        ),
        "weighted_population_sum": float(
            subset["WTSAF4YR"].sum()
        ),
        "strata": int(
            subset["pooled_stratum"].nunique()
        ),
        "psus": int(
            subset[
                [
                    "pooled_stratum",
                    "pooled_psu",
                ]
            ]
            .drop_duplicates()
            .shape[0]
        ),
    }


python_reconciliation = pd.DataFrame(
    [
        sample_summary(
            "canonical_full",
            pd.Series(
                True,
                index=cohort.index,
            ),
        ),
        sample_summary(
            "no_topcode",
            cohort[
                "no_topcode_sensitivity_eligible"
            ],
        ),
        sample_summary(
            "exclude_early_deaths",
            cohort[
                "early_death_exclusion_eligible"
            ],
        ),
    ]
)

display(python_reconciliation)
print(f"Model input written: {MODEL_INPUT_PATH}")


,sample,n,events,weighted_population_sum,strata,psus
0,canonical_full,4350,127,2.266872e+08,30,60
1,no_topcode,4079,88,2.176131e+08,30,60
2,exclude_early_deaths,4309,86,2.248156e+08,30,60


Model input written: <PROJECT_ROOT>\data\interim\10_mortality_model_input.csv


## 3. Run the governed R survey-Cox analysis

The R script uses the complex-survey Cox estimator and writes machine-readable coefficient, diagnostic, reconciliation, and package-version tables. It never installs packages automatically.


For compatibility across installed `survey` versions, the governed `svycoxph` call uses only the stable formula, design, and Efron-ties interface. The notebook captures and prints complete R stdout/stderr when execution fails.


The governed `svycoxph` call does not pass a `ties` argument because some installed `survey` versions interpret it as a model-frame variable and fail with `variable lengths differ (found for '(ties)')`. The package default partial-likelihood handling is therefore used for all survey-weighted Cox models. The separate diagnostic fallback `coxph` call retains explicit Efron handling.


The generated R source is parsed through a separate helper script instead of an inline `Rscript -e` expression. This avoids Windows path escaping and quoting issues. Any genuine parser failure is displayed with its R stderr tail.


The source guard now rejects only the exact previously observed malformed multiline lookup. Valid R double-bracket indexing such as `model_results[["canonical_primary_adjusted"]]` is accepted. The separate R parser remains the authoritative syntax check.


R does not concatenate adjacent string literals automatically. Multi-line error messages in the generated R script now use `paste0(...)`, and the notebook performs a source-level check for any remaining adjacent quoted strings before parsing.


In [5]:
R_SCRIPT_PATH = (
    SCRIPTS_ROOT
    / "10_mortality_survival_analysis.R"
)
R_MODEL_COEFFICIENTS_PATH = (
    TABLES_ROOT
    / "10_r_mortality_model_coefficients.csv"
)
R_MODEL_DIAGNOSTICS_PATH = (
    TABLES_ROOT
    / "10_r_mortality_model_diagnostics.csv"
)
R_PH_DIAGNOSTICS_PATH = (
    TABLES_ROOT
    / "10_r_mortality_ph_diagnostics.csv"
)
R_RECONCILIATION_PATH = (
    TABLES_ROOT
    / "10_r_mortality_reconciliation.csv"
)
R_VERSIONS_PATH = (
    TABLES_ROOT
    / "10_r_package_versions.csv"
)
R_STDOUT_PATH = (
    LOGS_ROOT
    / "10_mortality_r_stdout.txt"
)
R_STDERR_PATH = (
    LOGS_ROOT
    / "10_mortality_r_stderr.txt"
)

R_SCRIPT = r'''NOTEBOOK_BUILD_R <- "10-v8-r-string-concat-fixed"
cat(sprintf("R script build: %s\n", NOTEBOOK_BUILD_R))

args <- commandArgs(trailingOnly = TRUE)

if (length(args) != 6) {
  stop("Expected 6 command-line arguments.")
}

input_path <- args[[1]]
coefficients_path <- args[[2]]
diagnostics_path <- args[[3]]
ph_path <- args[[4]]
reconciliation_path <- args[[5]]
versions_path <- args[[6]]

required_packages <- c("survey", "survival")

missing_packages <- required_packages[
  !vapply(
    required_packages,
    requireNamespace,
    logical(1),
    quietly = TRUE
  )
]

if (length(missing_packages) > 0) {
  stop(
    paste0(
      "Missing required R packages: ",
      paste(missing_packages, collapse = ", "),
      ". Install with install.packages(c(",
      paste(
        sprintf('"%s"', missing_packages),
        collapse = ", "
      ),
      "))"
    )
  )
}

suppressPackageStartupMessages({
  library(survey)
  library(survival)
})

options(survey.lonely.psu = "fail")

df <- read.csv(
  input_path,
  stringsAsFactors = FALSE,
  check.names = FALSE
)

required_columns <- c(
  "SEQN",
  "NHANES_CYCLE",
  "chronological_age_years",
  "RIAGENDR",
  "RIDRETH3",
  "WTSAF4YR",
  "pooled_stratum",
  "pooled_psu",
  "phenoage_acceleration_per_5_years",
  "phenoage_acceleration_per_sd",
  "sensitivity_erratum_acceleration_per_5_years",
  "sensitivity_creatinine_plus_0_11_acceleration_per_5_years",
  "sensitivity_creatinine_plus_0_17_acceleration_per_5_years",
  "sensitivity_creatinine_plus_0_23_acceleration_per_5_years",
  "mortality_event",
  "followup_months",
  "no_topcode_sensitivity_eligible",
  "early_death_exclusion_eligible"
)

missing_columns <- setdiff(
  required_columns,
  names(df)
)

if (length(missing_columns) > 0) {
  stop(
    paste(
      "R model input is missing required columns:",
      paste(missing_columns, collapse = ", ")
    )
  )
}

numeric_columns <- c(
  "chronological_age_years",
  "RIAGENDR",
  "RIDRETH3",
  "WTSAF4YR",
  "phenoage_acceleration_per_5_years",
  "phenoage_acceleration_per_sd",
  "sensitivity_erratum_acceleration_per_5_years",
  "sensitivity_creatinine_plus_0_11_acceleration_per_5_years",
  "sensitivity_creatinine_plus_0_17_acceleration_per_5_years",
  "sensitivity_creatinine_plus_0_23_acceleration_per_5_years",
  "mortality_event",
  "followup_months"
)

for (column in numeric_columns) {
  df[[column]] <- as.numeric(df[[column]])
}

as_logical_safe <- function(x) {
  if (is.logical(x)) {
    x[is.na(x)] <- FALSE
    return(x)
  }

  text <- toupper(trimws(as.character(x)))
  result <- text %in% c(
    "TRUE",
    "T",
    "1",
    "YES",
    "Y"
  )
  result[is.na(result)] <- FALSE
  result
}

df$no_topcode_sensitivity_eligible <- as_logical_safe(
  df$no_topcode_sensitivity_eligible
)
df$early_death_exclusion_eligible <- as_logical_safe(
  df$early_death_exclusion_eligible
)

df$sex_factor <- factor(
  df$RIAGENDR,
  levels = sort(unique(df$RIAGENDR))
)
df$race_factor <- factor(
  df$RIDRETH3,
  levels = sort(unique(df$RIDRETH3))
)
df$cycle_factor <- factor(
  df$NHANES_CYCLE,
  levels = sort(unique(df$NHANES_CYCLE))
)
df$pooled_stratum <- factor(df$pooled_stratum)
df$pooled_psu <- factor(df$pooled_psu)

# PH diagnostics use a separate weighted Cox fit when
# cox.zph cannot operate directly on svycoxph. Normalizing
# weights preserves coefficients while avoiding numerical
# dependence on the raw NHANES weight magnitude.
df$ph_weight <- df$WTSAF4YR / mean(df$WTSAF4YR)

if (any(!is.finite(df$WTSAF4YR)) || any(df$WTSAF4YR <= 0)) {
  stop("R model input contains invalid survey weights.")
}

if (
  any(!is.finite(df$followup_months)) ||
  any(df$followup_months <= 0)
) {
  stop("R model input contains invalid follow-up.")
}

if (!all(df$mortality_event %in% c(0, 1))) {
  stop("R model input contains a non-binary event.")
}

full_design <- svydesign(
  ids = ~pooled_psu,
  strata = ~pooled_stratum,
  weights = ~WTSAF4YR,
  nest = TRUE,
  data = df
)

model_specs <- data.frame(
  model = c(
    "canonical_exposure_only",
    "canonical_primary_adjusted",
    "canonical_adjusted_per_sd",
    "sensitivity_no_topcode",
    "sensitivity_erratum",
    "sensitivity_creatinine_plus_0_11",
    "sensitivity_creatinine_plus_0_17",
    "sensitivity_creatinine_plus_0_23",
    "sensitivity_exclude_early_deaths"
  ),
  sample = c(
    "canonical_full",
    "canonical_full",
    "canonical_full",
    "no_topcode",
    "canonical_full",
    "canonical_full",
    "canonical_full",
    "canonical_full",
    "exclude_early_deaths"
  ),
  exposure = c(
    "phenoage_acceleration_per_5_years",
    "phenoage_acceleration_per_5_years",
    "phenoage_acceleration_per_sd",
    "phenoage_acceleration_per_5_years",
    "sensitivity_erratum_acceleration_per_5_years",
    "sensitivity_creatinine_plus_0_11_acceleration_per_5_years",
    "sensitivity_creatinine_plus_0_17_acceleration_per_5_years",
    "sensitivity_creatinine_plus_0_23_acceleration_per_5_years",
    "phenoage_acceleration_per_5_years"
  ),
  adjusted = c(
    FALSE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE,
    TRUE
  ),
  stringsAsFactors = FALSE
)

make_subdesign <- function(sample_name) {
  if (sample_name == "canonical_full") {
    return(full_design)
  }

  if (sample_name == "no_topcode") {
    return(
      subset(
        full_design,
        no_topcode_sensitivity_eligible
      )
    )
  }

  if (sample_name == "exclude_early_deaths") {
    return(
      subset(
        full_design,
        early_death_exclusion_eligible
      )
    )
  }

  stop(paste("Unknown sample:", sample_name))
}

make_formula <- function(exposure, adjusted) {
  terms <- exposure

  if (adjusted) {
    terms <- c(
      terms,
      "chronological_age_years",
      "sex_factor",
      "race_factor",
      "cycle_factor"
    )
  }

  as.formula(
    paste(
      "Surv(followup_months, mortality_event) ~",
      paste(terms, collapse = " + ")
    )
  )
}

extract_coefficients <- function(
  fit,
  model_name,
  sample_name,
  exposure_name,
  adjusted
) {
  beta <- coef(fit)
  covariance <- vcov(fit)
  standard_error <- sqrt(diag(covariance))
  z_value <- beta / standard_error
  p_value <- 2 * pnorm(
    abs(z_value),
    lower.tail = FALSE
  )

  data.frame(
    model = model_name,
    sample = sample_name,
    exposure = exposure_name,
    adjusted = adjusted,
    term = names(beta),
    coefficient = as.numeric(beta),
    standard_error = as.numeric(standard_error),
    z = as.numeric(z_value),
    p_value = as.numeric(p_value),
    hazard_ratio = exp(as.numeric(beta)),
    ci_low_95 = exp(
      as.numeric(beta) - 1.96 * as.numeric(standard_error)
    ),
    ci_high_95 = exp(
      as.numeric(beta) + 1.96 * as.numeric(standard_error)
    ),
    stringsAsFactors = FALSE
  )
}

fit_one_model <- function(spec_row) {
  model_name <- spec_row$model[[1]]
  sample_name <- spec_row$sample[[1]]
  exposure_name <- spec_row$exposure[[1]]
  adjusted <- spec_row$adjusted[[1]]

  design_object <- make_subdesign(sample_name)
  analysis_data <- droplevels(design_object$variables)
  formula_object <- make_formula(
    exposure_name,
    adjusted
  )
  warning_messages <- character(0)

  if (model_name == "canonical_exposure_only") {
    cat("Exact svycoxph call: svycoxph(formula_object, design = design_object)\n")
  }

  fit <- tryCatch(
    withCallingHandlers(
      svycoxph(
        formula_object,
        design = design_object
      ),
      warning = function(warning_condition) {
        warning_messages <<- c(
          warning_messages,
          conditionMessage(warning_condition)
        )
        invokeRestart("muffleWarning")
      }
    ),
    error = function(error_condition) {
      structure(
        list(
          message = conditionMessage(error_condition)
        ),
        class = "agelens_model_error"
      )
    }
  )

  psu_groups <- split(
    as.character(analysis_data$pooled_psu),
    as.character(analysis_data$pooled_stratum),
    drop = TRUE
  )
  psu_by_stratum <- vapply(
    psu_groups,
    function(values) length(unique(values)),
    integer(1)
  )

  if (inherits(fit, "agelens_model_error")) {
    diagnostics <- data.frame(
      model = model_name,
      sample = sample_name,
      exposure = exposure_name,
      adjusted = adjusted,
      n = nrow(analysis_data),
      events = sum(analysis_data$mortality_event),
      weighted_population_sum = sum(
        analysis_data$WTSAF4YR
      ),
      strata = length(psu_groups),
      psus = sum(psu_by_stratum),
      minimum_psus_per_stratum = min(psu_by_stratum),
      converged = FALSE,
      finite_coefficients = FALSE,
      finite_covariance = FALSE,
      fatal_warning = FALSE,
      warning_messages = paste(
        warning_messages,
        collapse = " | "
      ),
      error_message = fit$message,
      formula = paste(
        deparse(formula_object),
        collapse = ""
      ),
      stringsAsFactors = FALSE
    )

    return(
      list(
        fit = NULL,
        coefficients = NULL,
        diagnostics = diagnostics
      )
    )
  }

  beta <- coef(fit)
  covariance <- vcov(fit)

  fatal_warning <- any(
    grepl(
      "converg|infinite|singular|NaN|not positive definite",
      warning_messages,
      ignore.case = TRUE
    )
  )

  diagnostics <- data.frame(
    model = model_name,
    sample = sample_name,
    exposure = exposure_name,
    adjusted = adjusted,
    n = nrow(analysis_data),
    events = sum(analysis_data$mortality_event),
    weighted_population_sum = sum(
      analysis_data$WTSAF4YR
    ),
    strata = length(psu_groups),
    psus = sum(psu_by_stratum),
    minimum_psus_per_stratum = min(psu_by_stratum),
    converged = (
      all(is.finite(beta)) &&
      all(is.finite(covariance))
    ),
    finite_coefficients = all(is.finite(beta)),
    finite_covariance = all(is.finite(covariance)),
    fatal_warning = fatal_warning,
    warning_messages = paste(
      warning_messages,
      collapse = " | "
    ),
    error_message = "",
    formula = paste(
      deparse(formula_object),
      collapse = ""
    ),
    stringsAsFactors = FALSE
  )

  coefficients <- extract_coefficients(
    fit,
    model_name,
    sample_name,
    exposure_name,
    adjusted
  )

  list(
    fit = fit,
    coefficients = coefficients,
    diagnostics = diagnostics
  )
}

model_results <- vector(
  "list",
  nrow(model_specs)
)
names(model_results) <- model_specs$model

for (index in seq_len(nrow(model_specs))) {
  specification <- model_specs[index, , drop = FALSE]
  result <- fit_one_model(specification)
  model_results[[specification$model[[1]]]] <- result

  if (is.null(result$fit)) {
    stop(
      paste(
        "Model failed:",
        specification$model[[1]],
        result$diagnostics$error_message[[1]]
      )
    )
  }
}

coefficient_table <- do.call(
  rbind,
  lapply(
    model_results,
    function(result) result$coefficients
  )
)
diagnostic_table <- do.call(
  rbind,
  lapply(
    model_results,
    function(result) result$diagnostics
  )
)

primary_fit <- model_results[["canonical_primary_adjusted"]]$fit

ph_method <- "cox.zph_on_svycoxph"
ph_error <- ""
ph_result <- tryCatch(
  cox.zph(
    primary_fit,
    transform = "log",
    terms = TRUE,
    singledf = TRUE
  ),
  error = function(error_condition) {
    ph_error <<- conditionMessage(error_condition)
    NULL
  }
)

if (is.null(ph_result)) {
  ph_method <- "cox.zph_on_weighted_cluster_robust_coxph"
  primary_formula <- make_formula(
    "phenoage_acceleration_per_5_years",
    TRUE
  )

  fallback_fit <- coxph(
    primary_formula,
    data = df,
    weights = ph_weight,
    cluster = pooled_psu,
    robust = TRUE,
    ties = "efron",
    x = TRUE,
    model = TRUE
  )

  ph_result <- cox.zph(
    fallback_fit,
    transform = "log",
    terms = TRUE,
    singledf = TRUE
  )
}

ph_table_raw <- as.data.frame(ph_result$table)
ph_table_raw$term <- rownames(ph_table_raw)
rownames(ph_table_raw) <- NULL

chisq_column <- grep(
  "chisq",
  names(ph_table_raw),
  ignore.case = TRUE,
  value = TRUE
)[1]
p_column <- grep(
  "^p$|pvalue|p.value",
  names(ph_table_raw),
  ignore.case = TRUE,
  value = TRUE
)[1]

if (is.na(chisq_column) || is.na(p_column)) {
  stop(
    paste0(
      "Could not identify proportional-hazards ",
      "diagnostic columns."
    )
  )
}

ph_table <- data.frame(
  method = ph_method,
  term = ph_table_raw$term,
  chi_square = ph_table_raw[[chisq_column]],
  p_value = ph_table_raw[[p_column]],
  fallback_reason = ph_error,
  stringsAsFactors = FALSE
)

sample_names <- c(
  "canonical_full",
  "no_topcode",
  "exclude_early_deaths"
)

reconciliation_rows <- lapply(
  sample_names,
  function(sample_name) {
    design_object <- make_subdesign(sample_name)
    analysis_data <- droplevels(design_object$variables)
    psu_groups <- split(
      as.character(analysis_data$pooled_psu),
      as.character(analysis_data$pooled_stratum),
      drop = TRUE
    )
    psu_by_stratum <- vapply(
      psu_groups,
      function(values) length(unique(values)),
      integer(1)
    )

    data.frame(
      sample = sample_name,
      n = nrow(analysis_data),
      events = sum(analysis_data$mortality_event),
      weighted_population_sum = sum(
        analysis_data$WTSAF4YR
      ),
      strata = length(psu_groups),
      psus = sum(psu_by_stratum),
      stringsAsFactors = FALSE
    )
  }
)

reconciliation_table <- do.call(
  rbind,
  reconciliation_rows
)

version_table <- data.frame(
  component = c(
    "R",
    "survey",
    "survival"
  ),
  version = c(
    paste(
      R.version$major,
      R.version$minor,
      sep = "."
    ),
    as.character(
      packageVersion("survey")
    ),
    as.character(
      packageVersion("survival")
    )
  ),
  stringsAsFactors = FALSE
)

write.csv(
  coefficient_table,
  coefficients_path,
  row.names = FALSE,
  na = ""
)
write.csv(
  diagnostic_table,
  diagnostics_path,
  row.names = FALSE,
  na = ""
)
write.csv(
  ph_table,
  ph_path,
  row.names = FALSE,
  na = ""
)
write.csv(
  reconciliation_table,
  reconciliation_path,
  row.names = FALSE,
  na = ""
)
write.csv(
  version_table,
  versions_path,
  row.names = FALSE,
  na = ""
)

cat(
  sprintf(
    "Models completed: %d/%d\n",
    nrow(model_specs),
    nrow(model_specs)
  )
)
cat(
  sprintf(
    "Primary cohort: n=%d, events=%d\n",
    nrow(df),
    sum(df$mortality_event)
  )
)
cat(
  sprintf(
    "PH diagnostic method: %s\n",
    ph_method
  )
)
'''

expected_primary_lookup = (
    'primary_fit <- model_results'
    '[["canonical_primary_adjusted"]]$fit'
)

if expected_primary_lookup not in R_SCRIPT:
    raise RuntimeError(
        "Generated R source does not contain the corrected "
        "primary model lookup."
    )

# Reject only the previously observed malformed line break:
# model_results[
#   ["canonical_primary_adjusted"]
# ]
malformed_primary_lookup = (
    'primary_fit <- model_results[\n'
    '  ["canonical_primary_adjusted"]\n'
    ']$fit'
)

if malformed_primary_lookup in R_SCRIPT:
    raise RuntimeError(
        "Generated R source contains the old malformed "
        "primary model lookup."
    )

R_SCRIPT_PATH.write_text(
    R_SCRIPT,
    encoding="utf-8",
)

R_PARSE_HELPER_PATH = (
    SCRIPTS_ROOT
    / "10_parse_generated_r_script.R"
)

R_PARSE_HELPER = r'''args <- commandArgs(trailingOnly = TRUE)

if (length(args) != 1) {
  stop("Expected one R script path.")
}

target_path <- args[[1]]
parse(file = target_path)
cat("R parse check passed\n")
'''

R_PARSE_HELPER_PATH.write_text(
    R_PARSE_HELPER,
    encoding="utf-8",
)

parse_check = subprocess.run(
    [
        str(RSCRIPT_PATH),
        str(R_PARSE_HELPER_PATH),
        str(R_SCRIPT_PATH),
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)

print(parse_check.stdout)

if parse_check.returncode != 0:
    parse_stderr = parse_check.stderr.strip()
    print(parse_stderr)

    parse_tail = "\n".join(
        parse_stderr.splitlines()[-20:]
    )

    raise RuntimeError(
        "Generated R script failed static parse validation. "
        "R parse stderr tail:\n"
        f"{parse_tail}\n"
        f"Generated script: {R_SCRIPT_PATH}\n"
        f"Parse helper: {R_PARSE_HELPER_PATH}"
    )


command = [
    str(RSCRIPT_PATH),
    str(R_SCRIPT_PATH),
    str(MODEL_INPUT_PATH),
    str(R_MODEL_COEFFICIENTS_PATH),
    str(R_MODEL_DIAGNOSTICS_PATH),
    str(R_PH_DIAGNOSTICS_PATH),
    str(R_RECONCILIATION_PATH),
    str(R_VERSIONS_PATH),
]

print("Running governed R survey-Cox analysis...")

completed = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)

R_STDOUT_PATH.write_text(
    completed.stdout,
    encoding="utf-8",
)
R_STDERR_PATH.write_text(
    completed.stderr,
    encoding="utf-8",
)

print(completed.stdout)

if completed.returncode != 0:
    stderr_text = completed.stderr.strip()
    stdout_text = completed.stdout.strip()

    print("----- R STDOUT -----")
    print(
        stdout_text
        if stdout_text
        else "[empty]"
    )
    print("----- R STDERR -----")
    print(
        stderr_text
        if stderr_text
        else "[empty]"
    )

    stderr_tail = "\n".join(
        stderr_text.splitlines()[-20:]
    )

    raise RuntimeError(
        "The governed R survey-Cox analysis failed. "
        "R stderr tail:\n"
        f"{stderr_tail}\n"
        f"Full log: {R_STDERR_PATH}"
    )

required_r_outputs = [
    R_MODEL_COEFFICIENTS_PATH,
    R_MODEL_DIAGNOSTICS_PATH,
    R_PH_DIAGNOSTICS_PATH,
    R_RECONCILIATION_PATH,
    R_VERSIONS_PATH,
]

missing_r_outputs = [
    path
    for path in required_r_outputs
    if not path.exists()
]

if missing_r_outputs:
    raise RuntimeError(
        f"R completed without creating expected outputs: "
        f"{missing_r_outputs}"
    )

print("✅ Governed R survey-Cox analysis completed.")


expression(NOTEBOOK_BUILD_R <- "10-v8-r-string-concat-fixed", 
    cat(sprintf("R script build: %s\n", NOTEBOOK_BUILD_R)), args <- commandArgs(trailingOnly = TRUE), 
    if (length(args) != 6) {
        stop("Expected 6 command-line arguments.")
    }, input_path <- args[[1]], coefficients_path <- args[[2]], 
    diagnostics_path <- args[[3]], ph_path <- args[[4]], reconciliation_path <- args[[5]], 
    versions_path <- args[[6]], required_packages <- c("survey", 
        "survival"), missing_packages <- required_packages[!vapply(required_packages, 
        requireNamespace, logical(1), quietly = TRUE)], if (length(missing_packages) > 
        0) {
        stop(paste0("Missing required R packages: ", paste(missing_packages, 
            collapse = ", "), ". Install with install.packages(c(", 
            paste(sprintf("\"%s\"", missing_packages), collapse = ", "), 
            "))"))
    }, suppressPackageStartupMessages({
        library(survey)
        library(survival)
    }), optio

## 4. Reconcile and validate mortality models

In [6]:
coefficients = pd.read_csv(
    R_MODEL_COEFFICIENTS_PATH
)
model_diagnostics = pd.read_csv(
    R_MODEL_DIAGNOSTICS_PATH
)
ph_diagnostics = pd.read_csv(
    R_PH_DIAGNOSTICS_PATH
)
r_reconciliation = pd.read_csv(
    R_RECONCILIATION_PATH
)
r_versions = pd.read_csv(
    R_VERSIONS_PATH
)

for boolean_column in [
    "adjusted",
    "converged",
    "finite_coefficients",
    "finite_covariance",
    "fatal_warning",
]:
    model_diagnostics[
        boolean_column
    ] = parse_r_boolean(
        model_diagnostics[boolean_column]
    )

coefficients["adjusted"] = parse_r_boolean(
    coefficients["adjusted"]
)

required_models = [
    "canonical_exposure_only",
    "canonical_primary_adjusted",
    "canonical_adjusted_per_sd",
    "sensitivity_no_topcode",
    "sensitivity_erratum",
    "sensitivity_creatinine_plus_0_11",
    "sensitivity_creatinine_plus_0_17",
    "sensitivity_creatinine_plus_0_23",
    "sensitivity_exclude_early_deaths",
]

exposure_results = coefficients.loc[
    coefficients["term"].eq(
        coefficients["exposure"]
    )
].copy()

model_exposure_lookup = (
    exposure_results.set_index("model")
)

check_records = []


def record_check(
    check: str,
    passed: bool,
    observed: object,
    expected: object,
    tolerance: object = None,
) -> None:
    check_records.append(
        {
            "check": check,
            "pass": bool(passed),
            "observed": observed,
            "expected": expected,
            "tolerance": tolerance,
        }
    )


record_check(
    "All required models are present",
    set(required_models)
    == set(model_diagnostics["model"]),
    sorted(model_diagnostics["model"].tolist()),
    sorted(required_models),
)

record_check(
    "Each model has an exposure coefficient",
    set(required_models)
    == set(exposure_results["model"]),
    sorted(exposure_results["model"].tolist()),
    sorted(required_models),
)

record_check(
    "All models converged",
    model_diagnostics["converged"].all(),
    model_diagnostics.loc[
        ~model_diagnostics["converged"],
        "model",
    ].tolist(),
    [],
)

record_check(
    "All model coefficients are finite",
    np.isfinite(
        coefficients[
            [
                "coefficient",
                "standard_error",
                "hazard_ratio",
                "ci_low_95",
                "ci_high_95",
                "p_value",
            ]
        ].to_numpy(dtype=float)
    ).all(),
    "finite",
    "finite",
)

record_check(
    "No fatal model warnings occurred",
    not model_diagnostics[
        "fatal_warning"
    ].any(),
    model_diagnostics.loc[
        model_diagnostics["fatal_warning"],
        [
            "model",
            "warning_messages",
        ],
    ].to_dict("records"),
    [],
)

record_check(
    "No R model errors occurred",
    model_diagnostics[
        "error_message"
    ].fillna("").eq("").all(),
    model_diagnostics.loc[
        ~model_diagnostics[
            "error_message"
        ].fillna("").eq(""),
        [
            "model",
            "error_message",
        ],
    ].to_dict("records"),
    [],
)

record_check(
    "Every model retains at least two PSUs per stratum",
    model_diagnostics[
        "minimum_psus_per_stratum"
    ].ge(2).all(),
    float(
        model_diagnostics[
            "minimum_psus_per_stratum"
        ].min()
    ),
    2,
    ">=",
)

reconciliation = python_reconciliation.merge(
    r_reconciliation,
    on="sample",
    suffixes=("_python", "_r"),
    validate="one_to_one",
)

for column in [
    "n",
    "events",
    "strata",
    "psus",
]:
    record_check(
        f"Python and R {column} reconcile",
        reconciliation[
            f"{column}_python"
        ].eq(
            reconciliation[
                f"{column}_r"
            ]
        ).all(),
        reconciliation[
            [
                "sample",
                f"{column}_python",
                f"{column}_r",
            ]
        ].to_dict("records"),
        "exact equality",
    )

maximum_weight_difference = float(
    np.max(
        np.abs(
            reconciliation[
                "weighted_population_sum_python"
            ]
            - reconciliation[
                "weighted_population_sum_r"
            ]
        )
    )
)

record_check(
    "Python and R weighted sums reconcile",
    maximum_weight_difference <= 1e-4,
    maximum_weight_difference,
    0.0,
    1e-4,
)

primary_ph_rows = ph_diagnostics.loc[
    ph_diagnostics["term"].eq(
        "phenoage_acceleration_per_5_years"
    )
]

record_check(
    "Primary PH diagnostic is present",
    len(primary_ph_rows) == 1,
    primary_ph_rows[
        [
            "method",
            "term",
            "p_value",
        ]
    ].to_dict("records"),
    "one exposure row",
)

primary_ph_p = (
    float(primary_ph_rows.iloc[0]["p_value"])
    if len(primary_ph_rows) == 1
    else np.nan
)

record_check(
    "Primary acceleration passes PH threshold",
    bool(
        np.isfinite(primary_ph_p)
        and primary_ph_p >= 0.05
    ),
    primary_ph_p,
    0.05,
    ">=",
)

canonical_adjusted = model_exposure_lookup.loc[
    "canonical_primary_adjusted"
]

for model_name in [
    "sensitivity_creatinine_plus_0_11",
    "sensitivity_creatinine_plus_0_17",
    "sensitivity_creatinine_plus_0_23",
]:
    sensitivity_row = model_exposure_lookup.loc[
        model_name
    ]

    coefficient_difference = abs(
        float(
            sensitivity_row["coefficient"]
        )
        - float(
            canonical_adjusted["coefficient"]
        )
    )
    se_difference = abs(
        float(
            sensitivity_row["standard_error"]
        )
        - float(
            canonical_adjusted["standard_error"]
        )
    )

    record_check(
        f"{model_name} coefficient equals canonical",
        coefficient_difference <= 1e-10,
        coefficient_difference,
        0.0,
        1e-10,
    )
    record_check(
        f"{model_name} SE equals canonical",
        se_difference <= 1e-10,
        se_difference,
        0.0,
        1e-10,
    )

formula_text = " ".join(
    model_diagnostics[
        "formula"
    ].fillna("").astype(str).tolist()
)

cause_specific_hits = [
    token
    for token in [
        "UCOD",
        "DIABETES",
        "HYPERTEN",
        "cause",
    ]
    if token.lower() in formula_text.lower()
]

record_check(
    "No cause-specific outcome was modeled",
    len(cause_specific_hits) == 0,
    cause_specific_hits,
    [],
)

record_check(
    "Primary adjusted result is finite",
    np.isfinite(
        canonical_adjusted[
            [
                "hazard_ratio",
                "ci_low_95",
                "ci_high_95",
                "p_value",
            ]
        ].to_numpy(dtype=float)
    ).all(),
    canonical_adjusted[
        [
            "hazard_ratio",
            "ci_low_95",
            "ci_high_95",
            "p_value",
        ]
    ].to_dict(),
    "finite",
)

release_checks = pd.DataFrame(
    check_records
)

if not release_checks["pass"].all():
    failure_path = (
        TABLES_ROOT
        / "10_mortality_release_failures.csv"
    )

    release_checks.loc[
        ~release_checks["pass"]
    ].to_csv(
        failure_path,
        index=False,
    )

    display(
        release_checks.loc[
            ~release_checks["pass"]
        ]
    )

    raise RuntimeError(
        "Mortality model validation failed. "
        "Results remain blocked. "
        f"Failure audit: {failure_path}"
    )

display(
    exposure_results[
        [
            "model",
            "sample",
            "hazard_ratio",
            "ci_low_95",
            "ci_high_95",
            "p_value",
        ]
    ].round(6)
)
display(ph_diagnostics.round(6))
display(release_checks)

print(
    f"✅ Mortality model validation passed: "
    f"{len(release_checks)}/{len(release_checks)}"
)


,model,sample,hazard_ratio,ci_low_95,ci_high_95,p_value
0,canonical_exposure_only,canonical_full,1.232435,1.185107,1.281652,0.0
1,canonical_primary_adjusted,canonical_full,1.185354,1.128633,1.244926,0.0
10,canonical_adjusted_per_sd,canonical_full,1.284997,1.195360,1.381357,0.0
19,sensitivity_no_topcode,no_topcode,1.179104,1.117874,1.243687,0.0
28,sensitivity_erratum,canonical_full,1.188679,1.130885,1.249426,0.0
37,sensitivity_creatinine_plus_0_11,canonical_full,1.185354,1.128633,1.244926,0.0
46,sensitivity_creatinine_plus_0_17,canonical_full,1.185354,1.128633,1.244926,0.0
55,sensitivity_creatinine_plus_0_23,canonical_full,1.185354,1.128633,1.244926,0.0
64,sensitivity_exclude_early_deaths,exclude_early_deaths,1.176648,1.104732,1.253245,0.0


,method,term,chi_square,p_value,fallback_reason
0,cox.zph_on_svycoxph,phenoage_acceleration_per_5_years,0.000003,0.998565,NaN
1,cox.zph_on_svycoxph,chronological_age_years,0.001656,0.967539,NaN
2,cox.zph_on_svycoxph,sex_factor,0.000002,0.998771,NaN
3,cox.zph_on_svycoxph,race_factor,0.000490,0.982340,NaN
4,cox.zph_on_svycoxph,cycle_factor,0.001649,0.967605,NaN
5,cox.zph_on_svycoxph,GLOBAL,0.004480,1.000000,NaN


,check,pass,observed,expected,tolerance
0,All required models are present,True,"[canonical_adjusted_per_sd, canonical_exposure...","[canonical_adjusted_per_sd, canonical_exposure...",None
1,Each model has an exposure coefficient,True,"[canonical_adjusted_per_sd, canonical_exposure...","[canonical_adjusted_per_sd, canonical_exposure...",None
2,All models converged,True,[],[],None
3,All model coefficients are finite,True,finite,finite,None
4,No fatal model warnings occurred,True,[],[],None
5,No R model errors occurred,True,[],[],None
6,Every model retains at least two PSUs per stratum,True,2.0,2,>=
7,Python and R n reconcile,True,"[{'sample': 'canonical_full', 'n_python': 4350...",exact equality,None
8,Python and R events reconcile,True,"[{'sample': 'canonical_full', 'events_python':...",exact equality,None
9,Python and R strata reconcile,True,"[{'sample': 'canonical_full', 'strata_python':...",exact equality,None


✅ Mortality model validation passed: 22/22


## 5. Write validation artifacts and open the release gate

In [7]:
EXPOSURE_RESULTS_PATH = (
    TABLES_ROOT
    / "10_mortality_exposure_results.csv"
)
COEFFICIENTS_PATH = (
    TABLES_ROOT
    / "10_mortality_model_coefficients.csv"
)
MODEL_DIAGNOSTICS_PATH = (
    TABLES_ROOT
    / "10_mortality_model_diagnostics.csv"
)
PH_DIAGNOSTICS_PATH = (
    TABLES_ROOT
    / "10_mortality_ph_diagnostics.csv"
)
RECONCILIATION_PATH = (
    TABLES_ROOT
    / "10_mortality_reconciliation.csv"
)
RELEASE_CHECKS_PATH = (
    TABLES_ROOT
    / "10_mortality_release_checks.csv"
)
WRITE_AUDIT_PATH = (
    TABLES_ROOT
    / "10_mortality_release_write_audit.csv"
)
REPORT_PATH = (
    REPORT_ROOT
    / "Mortality_Model_Validation_Report.md"
)
METADATA_PATH = (
    LOGS_ROOT
    / "10_mortality_survival_analysis_metadata.json"
)

exposure_results.to_csv(
    EXPOSURE_RESULTS_PATH,
    index=False,
)
coefficients.to_csv(
    COEFFICIENTS_PATH,
    index=False,
)
model_diagnostics.to_csv(
    MODEL_DIAGNOSTICS_PATH,
    index=False,
)
ph_diagnostics.to_csv(
    PH_DIAGNOSTICS_PATH,
    index=False,
)
reconciliation.to_csv(
    RECONCILIATION_PATH,
    index=False,
)
release_checks.to_csv(
    RELEASE_CHECKS_PATH,
    index=False,
)

primary_row = model_exposure_lookup.loc[
    "canonical_primary_adjusted"
]

primary_hr = float(primary_row["hazard_ratio"])
primary_ci_low = float(primary_row["ci_low_95"])
primary_ci_high = float(primary_row["ci_high_95"])
primary_p = float(primary_row["p_value"])

report_results = exposure_results.loc[
    :,
    [
        "model",
        "sample",
        "hazard_ratio",
        "ci_low_95",
        "ci_high_95",
        "p_value",
    ],
].copy()

report = f"""# AgeLens V1 Mortality Model Validation Report

## Status

All required survey-weighted Cox models and release checks passed.

## Authorized Cohort

- Participants: {len(cohort):,}
- Deaths: {int(cohort['mortality_event'].sum()):,}
- Weighted population sum: {cohort['WTSAF4YR'].sum():.6f}
- Survey strata: {cohort['pooled_stratum'].nunique()}
- Survey PSUs: {cohort[['pooled_stratum', 'pooled_psu']].drop_duplicates().shape[0]}

## Primary Result

The primary adjusted hazard ratio per 5-year higher canonical Phenotypic Age acceleration was:

- HR: {primary_hr:.6f}
- 95% CI: {primary_ci_low:.6f} to {primary_ci_high:.6f}
- p-value: {primary_p:.6g}

## Exposure Results

{dataframe_to_markdown(report_results.round(6))}

## Proportional-Hazards Diagnostic

{dataframe_to_markdown(ph_diagnostics.round(6))}

## Reconciliation

{dataframe_to_markdown(reconciliation.round(6))}

## R Environment

{dataframe_to_markdown(r_versions)}

## Validation

All {len(release_checks)} release checks passed.

Cause-specific mortality was not modeled.

{RELEASE_MARKER}
"""

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

decision_text = replace_document_field(
    decision_text,
    "Version",
    "1.9",
)
decision_text = replace_document_field(
    decision_text,
    "Last Updated",
    RELEASE_DATE,
)
decision_text = add_revision_row(
    decision_text,
    (
        "| 1.9 | 2026-07-22 | Approved D-016: "
        "mortality models completed, validated, reconciled, "
        "and released for reporting. |"
    ),
)

decision_entry = f"""
{RELEASE_MARKER}

---

### D-016

| Field | Value |
| --- | --- |
| Title | Release AgeLens V1 all-cause mortality model results |
| Description | Release the D-015-authorized survey-weighted Cox results after all required models, sensitivity analyses, reconciliation checks, and proportional-hazards diagnostics passed. |
| Related Research Question | RQ4, RQ5 |
| Supporting Evidence | Authorized cohort: {len(cohort):,} participants and {int(cohort['mortality_event'].sum()):,} deaths. Primary adjusted HR per 5-year higher acceleration: {primary_hr:.6f} (95% CI {primary_ci_low:.6f}–{primary_ci_high:.6f}; p={primary_p:.6g}). PH diagnostic p={primary_ph_p:.6g}. |
| Evidence Level | Direct governed model execution and validation |
| Confidence Rating | High for the prespecified all-cause mortality analysis within the public-use follow-up window |
| Reviewer | Project owner |
| Status | Approved |
| Date | 2026-07-22 |
| Related Assumptions | D-015 cohort and model specification |
| Related Evidence Gaps | None blocking release |
| Notes | All required sensitivity models were completed. The three creatinine-shift acceleration models reproduced the canonical model because cycle-specific residualization removes constant additive shifts. Cause-specific mortality remains unauthorized. |
"""

decision_text = (
    decision_text.rstrip()
    + "\n\n"
    + decision_entry.strip()
    + "\n"
)

protocol_text = replace_document_field(
    protocol_text,
    "Version",
    "1.1",
)
protocol_text = replace_document_field(
    protocol_text,
    "Status",
    "Implemented and validated",
)
protocol_text = replace_document_field(
    protocol_text,
    "Last Updated",
    RELEASE_DATE,
)

ph_method = str(
    primary_ph_rows.iloc[0]["method"]
)

implementation_section = f"""

## 11. Implementation and Validation

Notebook 10 completed all nine required survey-weighted Cox models.

Primary adjusted HR per 5-year higher canonical acceleration:

- HR: {primary_hr:.6f}
- 95% CI: {primary_ci_low:.6f} to {primary_ci_high:.6f}
- p-value: {primary_p:.6g}

The proportional-hazards diagnostic used `{ph_method}` and returned `p = {primary_ph_p:.6g}` for the primary acceleration term.

Python and R cohort counts, event counts, weighted sums, strata, and PSUs reconciled within the governed tolerances.

D-016 releases the prespecified all-cause mortality results for reporting. Cause-specific mortality remains unauthorized.

{RELEASE_MARKER}
"""

protocol_text = (
    protocol_text.rstrip()
    + implementation_section
    + "\n"
)

if extract_document_version(decision_text) != "1.9":
    raise RuntimeError(
        "Updated Decision Log version is not 1.9."
    )

if f"### {DECISION_ID}" not in decision_text:
    raise RuntimeError(
        "D-016 was not constructed."
    )

if extract_document_version(protocol_text) != "1.1":
    raise RuntimeError(
        "Updated mortality protocol version is not 1.1."
    )

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

BACKUP_ROOT = (
    LOGS_ROOT
    / "mortality_release_backups"
    / timestamp
)
BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

backup_specs = {
    "decision_log": DECISION_LOG_PATH,
    "mortality_protocol": PROTOCOL_PATH,
    "config": CONFIG_PATH,
}

write_audit_records = []

for artifact, source_path in backup_specs.items():
    backup_path = (
        BACKUP_ROOT / source_path.name
    )
    shutil.copy2(
        source_path,
        backup_path,
    )

    before_hash = sha256_file(
        source_path
    )
    backup_hash = sha256_file(
        backup_path
    )

    if before_hash != backup_hash:
        raise RuntimeError(
            f"Backup hash mismatch for {artifact}."
        )

    write_audit_records.append(
        {
            "artifact": artifact,
            "path": str(
                source_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "backup": str(
                backup_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "sha256_before": before_hash,
            "sha256_backup": backup_hash,
            "sha256_after": "",
        }
    )

DECISION_LOG_PATH.write_text(
    decision_text,
    encoding="utf-8",
)
PROTOCOL_PATH.write_text(
    protocol_text,
    encoding="utf-8",
)

updated_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

updated_config["project"]["status"] = (
    "mortality_v1_results_ready"
)
updated_config["governance"][
    "mortality_results_release_decision"
] = DECISION_ID

updated_config["mortality_analysis"][
    "status"
] = "completed_validated_and_released"
updated_config["mortality_analysis"][
    "results_release_decision"
] = DECISION_ID
updated_config["mortality_analysis"][
    "primary_result"
] = {
    "scale": (
        "hazard ratio per 5-year higher "
        "Phenotypic Age acceleration"
    ),
    "hazard_ratio": primary_hr,
    "ci_low_95": primary_ci_low,
    "ci_high_95": primary_ci_high,
    "p_value": primary_p,
    "ph_diagnostic_p_value": primary_ph_p,
}
updated_config["mortality_analysis"][
    "validation_report"
] = str(
    REPORT_PATH.relative_to(
        PROJECT_ROOT
    )
)

updated_config["release_gates"][
    "mortality_models_completed"
] = True
updated_config["release_gates"][
    "mortality_model_validation_passed"
] = True
updated_config["release_gates"][
    "mortality_results_allowed"
] = True

updated_config["release_scope"][
    "mortality_analysis"
] = "completed_and_validated"
updated_config["release_scope"][
    "mortality_results"
] = "authorized"
updated_config["release_scope"][
    "cause_specific_mortality"
] = "not_authorized"

CONFIG_PATH.write_text(
    json.dumps(
        updated_config,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

for record in write_audit_records:
    source_path = (
        PROJECT_ROOT / record["path"]
    )
    after_hash = sha256_file(
        source_path
    )
    record["sha256_after"] = after_hash

    if after_hash == record["sha256_before"]:
        raise RuntimeError(
            f"{record['artifact']} did not change."
        )

write_audit = pd.DataFrame(
    write_audit_records
)
write_audit.to_csv(
    WRITE_AUDIT_PATH,
    index=False,
)

reloaded_config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

final_gate_checks = pd.DataFrame(
    [
        {
            "check": "Mortality models completed",
            "pass": reloaded_config[
                "release_gates"
            ][
                "mortality_models_completed"
            ],
        },
        {
            "check": "Mortality model validation passed",
            "pass": reloaded_config[
                "release_gates"
            ][
                "mortality_model_validation_passed"
            ],
        },
        {
            "check": "Mortality results authorized",
            "pass": reloaded_config[
                "release_gates"
            ][
                "mortality_results_allowed"
            ],
        },
        {
            "check": "D-016 recorded",
            "pass": reloaded_config[
                "governance"
            ][
                "mortality_results_release_decision"
            ]
            == "D-016",
        },
        {
            "check": "Cause-specific mortality remains unauthorized",
            "pass": (
                reloaded_config[
                    "release_scope"
                ][
                    "cause_specific_mortality"
                ]
                == "not_authorized"
            ),
        },
    ]
)

if not final_gate_checks["pass"].all():
    display(
        final_gate_checks.loc[
            ~final_gate_checks["pass"]
        ]
    )
    raise RuntimeError(
        "Final mortality release-gate verification failed."
    )

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "10_mortality_survival_analysis.ipynb"
    ),
    "authorization_decision": "D-015",
    "release_decision": "D-016",
    "cohort_rows": int(len(cohort)),
    "death_count": int(
        cohort["mortality_event"].sum()
    ),
    "models_completed": int(
        len(required_models)
    ),
    "release_checks_passed": int(
        release_checks["pass"].sum()
    ),
    "release_checks_total": int(
        len(release_checks)
    ),
    "primary_result": {
        "hazard_ratio": primary_hr,
        "ci_low_95": primary_ci_low,
        "ci_high_95": primary_ci_high,
        "p_value": primary_p,
        "ph_diagnostic_p_value": primary_ph_p,
    },
    "mortality_results_allowed": True,
    "cause_specific_mortality_authorized": False,
    "r_versions": r_versions.to_dict(
        orient="records"
    ),
    "backup_root": str(
        BACKUP_ROOT.relative_to(
            PROJECT_ROOT
        )
    ),
}

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

display(final_gate_checks)
display(write_audit)

print("✅ D-016 applied.")
print("✅ Mortality V1 model validation completed.")
print("✅ Prespecified all-cause mortality results are reportable.")
print("Cause-specific mortality remains unauthorized.")
print(
    "Primary adjusted HR per 5-year acceleration: "
    f"{primary_hr:.6f} "
    f"(95% CI {primary_ci_low:.6f}–{primary_ci_high:.6f}; "
    f"p={primary_p:.6g})"
)
print(f"Configuration backup: {BACKUP_ROOT}")


,check,pass
0,Mortality models completed,True
1,Mortality model validation passed,True
2,Mortality results authorized,True
3,D-016 recorded,True
4,Cause-specific mortality remains unauthorized,True


,artifact,path,backup,sha256_before,sha256_backup,sha256_after
0,decision_log,docs\governance\Decision_Log.md,logs\mortality_release_backups\20260722T171431...,8f20c72180cc78e8b085298aad9c90b020daea1fe5711d...,8f20c72180cc78e8b085298aad9c90b020daea1fe5711d...,0e6f4ace25e3c8bcbfe1b31ca37f89710515e672e6c990...
1,mortality_protocol,docs\methodology\Mortality_Analysis_Protocol.md,logs\mortality_release_backups\20260722T171431...,9990eade097d73dc645ba77c681e0e26ce518da5670332...,9990eade097d73dc645ba77c681e0e26ce518da5670332...,a5f39b8fec6d17d88021efb08c04022a575ffae0eca08d...
2,config,configs\agelens_config.json,logs\mortality_release_backups\20260722T171431...,82fab97bf864c05c8ccd7c24bbc71bbbd2c485bc53afd8...,82fab97bf864c05c8ccd7c24bbc71bbbd2c485bc53afd8...,cea65621f9c13ffa4b25dbaa3b1a310abb3fb7cbbb33b2...


✅ D-016 applied.
✅ Mortality V1 model validation completed.
✅ Prespecified all-cause mortality results are reportable.
Cause-specific mortality remains unauthorized.
Primary adjusted HR per 5-year acceleration: 1.185354 (95% CI 1.128633–1.244926; p=1.06871e-11)
Configuration backup: <PROJECT_ROOT>\logs\mortality_release_backups\20260722T171431Z
